# 12b · Sensibilidad del presupuesto Exact-K

Este cuaderno evalúa si las conclusiones estructurales y la recuperación
biológica Basal-like se mantienen al variar el número de aristas exportadas.
Reutiliza las puntuaciones densas ya producidas por los entrenamientos y **no
reentrena**, **no selecciona modelos** y **no abre cohortes externas**.

Se estudia la rejilla predefinida

\[
\kappa\in\{0.007,0.010,0.015,0.020\},\qquad
K=\operatorname{round}(\kappa |E_{prior}|),
\]

que para 52.878 aristas candidatas equivale a 370, 529, 793 y 1.058
aristas. En cada valor se vuelve a aplicar la misma selección modular
determinista; no se truncan los CSV de 529 aristas.

El análisis mantiene separados dos estimandos:

1. **Estabilidad descriptiva de cinco semillas:** Jaccard, índice de
   Kuncheva, estabilidad de Nogueira, núcleo 5/5 y topología.
2. **Prueba por permutación emparejada:** observado y cada permutación usan
   exactamente las dos mismas semillas. La estadística primaria es el
   margen Basal-like de enriquecimiento emparejado por el prior, y la
   estadística robusta es el peor margen en toda la rejilla,
   \(T_{rob}=\min_K S(K)\).

El valor *p* empírico se calcula como
\((1+\#\{T_b\geq T_{obs}\})/(B+1)\). El resultado solo se etiqueta como
completo cuando están disponibles las 49 permutaciones previstas.


In [ ]:
from __future__ import annotations

from functools import lru_cache
from hashlib import sha256
from itertools import combinations
from pathlib import Path
from textwrap import dedent
import json
import math
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/intergate-mpl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print

    def Markdown(value):
        return value


HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / "intergate").is_dir()
        and (path / "configs" / "protocol_v8.json").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encuentra la raíz del repositorio. Abra este notebook "
        "desde el directorio de InterGATE o uno de sus subdirectorios."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.biology import combine_biology_scores, module_aware_exact_k
from intergate.posthoc_biology_common import (
    load_locked_signature_resources,
    safe_name,
)
from intergate.revision_analyses import aggregate_seed_graphs, topology_metrics


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


## 1. Configuración

En condiciones normales no hay que editar nada. Las rutas pueden
sobrescribirse con `INTERGATE_RESULTS_ROOT`, `INTERGATE_ARTIFACTS_ROOT`,
`INTERGATE_OBSERVED_DENSE_ROOT` e `INTERGATE_BIOLOGY_RESOURCES`.

El cuaderno lee únicamente artefactos de desarrollo. Las carpetas de
datos externos no forman parte de ninguna ruta candidata.


In [ ]:
def first_existing(values, *, predicate=lambda path: path.exists(), label="ruta"):
    candidates = [
        Path(value).expanduser().resolve()
        for value in values
        if value is not None and str(value).strip()
    ]
    for path in candidates:
        if predicate(path):
            return path
    raise FileNotFoundError(
        f"No se encontró {label}. Candidatas:\n- "
        + "\n- ".join(map(str, candidates))
    )


def sha256_file(path):
    digest = sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


PROTOCOL_FILE = PROJECT_ROOT / "configs" / "protocol_v8.json"
PROTOCOL = json.loads(PROTOCOL_FILE.read_text(encoding="utf-8"))
LOCKED_SEEDS = tuple(map(int, PROTOCOL["outer_seeds"]))
PERMUTATION_MODEL_SEEDS = tuple(
    map(
        int,
        PROTOCOL.get("validation", {}).get(
            "label_permutation_model_seeds", (1234, 2025)
        ),
    )
)
N_TARGET_PERMUTATIONS = int(
    PROTOCOL.get("validation", {}).get("label_permutation_count", 49)
)
CLASS_NAMES = tuple(map(str, PROTOCOL["endpoint"]))
TARGET_CLASS = "Basal-like"
if TARGET_CLASS not in CLASS_NAMES:
    raise RuntimeError(f"{TARGET_CLASS!r} no está en el endpoint bloqueado.")

RESULTS_ROOT = first_existing(
    [
        os.environ.get("INTERGATE_RESULTS_ROOT"),
        PROJECT_ROOT / "results_v8",
        PROJECT_ROOT.parent / "locked_results_v8",
    ],
    predicate=lambda path: (
        path / "05_biology_validation" / "candidate_prior_for_null.csv.gz"
    ).is_file(),
    label="la raíz de resultados",
)

try:
    from intergate.config import CFG

    configured_artifacts = Path(CFG.ARTIFACTS_ROOT)
    configured_observed_name = str(
        getattr(CFG, "V8_ARTIFACT_NAME", "INTERGATE_V8")
    )
except Exception:
    CFG = None
    configured_artifacts = None
    configured_observed_name = "INTERGATE_V8"

ARTIFACTS_ROOT = first_existing(
    [
        os.environ.get("INTERGATE_ARTIFACTS_ROOT"),
        configured_artifacts,
        PROJECT_ROOT / "artifacts" / "intrinsic4",
        PROJECT_ROOT / "artifacts",
    ],
    predicate=lambda path: path.is_dir(),
    label="la raíz de artefactos densos",
)
OBSERVED_DENSE_ROOT = first_existing(
    [
        os.environ.get("INTERGATE_OBSERVED_DENSE_ROOT"),
        ARTIFACTS_ROOT / configured_observed_name,
        ARTIFACTS_ROOT / "INTERGATE_V8",
    ],
    predicate=lambda path: all(
        (path / f"seed_{seed}" / "transport_support_dense.npz").is_file()
        for seed in LOCKED_SEEDS
    ),
    label="los cinco artefactos observados transport_support_dense.npz",
)

resource_candidates = [
    value
    for value in (
        os.environ.get("INTERGATE_BIOLOGY_RESOURCES"),
        PROJECT_ROOT / "resources_biological",
        RESULTS_ROOT.parent / "resources_biological",
    )
    if value is not None
]
RESOURCE_DIR = first_existing(
    resource_candidates,
    predicate=lambda path: (
        (path / "PAM50_symbols.txt").is_file()
        and any((path / "msigdb").glob("c2.cgp.v*.symbols.gmt"))
    ),
    label="resources_biological con MSigDB y PAM50",
)

REVIEW_ROOT = RESULTS_ROOT / "12_reviewer_controls"
PERMUTATION_RESULT_ROOT = REVIEW_ROOT / "label_permutation_retraining"
OUTPUT_DIR = REVIEW_ROOT / "k_budget_sensitivity"
FIGURE_DIR = OUTPUT_DIR / "figures"
SEED_EXPORT_DIR = OUTPUT_DIR / "observed_seed_cuts"
CONSENSUS_EXPORT_DIR = OUTPUT_DIR / "observed_consensus"
for path in (OUTPUT_DIR, FIGURE_DIR, SEED_EXPORT_DIR, CONSENSUS_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

KAPPA_GRID = tuple(
    float(value)
    for value in os.environ.get(
        "INTERGATE_KAPPA_GRID", "0.007,0.010,0.015,0.020"
    ).split(",")
)
if not KAPPA_GRID or any(value <= 0 or value >= 1 for value in KAPPA_GRID):
    raise ValueError("INTERGATE_KAPPA_GRID debe contener fracciones en (0,1).")
if len(set(KAPPA_GRID)) != len(KAPPA_GRID):
    raise ValueError("La rejilla de kappa contiene duplicados.")

BIOLOGY_GRAPH = PROTOCOL.get("biology_graph", {})
SCORE_TERMS = BIOLOGY_GRAPH.get("definitive_locked_score_terms") or {
    "absolute_centered_class_delta": 1.0,
    "signed_context_agreement": 0.35,
    "cohort_support": 0.075,
}
CONTEXT_WEIGHT = float(SCORE_TERMS["signed_context_agreement"])
SUPPORT_WEIGHT = float(SCORE_TERMS["cohort_support"])
MODULE_POOL_MULTIPLIER = int(BIOLOGY_GRAPH.get("module_pool_multiplier", 6))
MODULE_TARGET_COMPONENTS = int(BIOLOGY_GRAPH.get("target_components", 12))
MODULE_MIN_EDGES = int(
    BIOLOGY_GRAPH.get("minimum_edges_per_seed_module", 3)
)
PERMUTATION_ARTIFACT_PREFIX = os.environ.get(
    "INTERGATE_PERMUTATION_ARTIFACT_PREFIX", "INTERGATE_V8_LABELPERM_"
)

configuration = pd.Series({
    "project_root": str(PROJECT_ROOT),
    "results_root": str(RESULTS_ROOT),
    "artifacts_root": str(ARTIFACTS_ROOT),
    "observed_dense_root": str(OBSERVED_DENSE_ROOT),
    "resource_dir": str(RESOURCE_DIR),
    "output_dir": str(OUTPUT_DIR),
    "classes": CLASS_NAMES,
    "five_seed_descriptive": LOCKED_SEEDS,
    "paired_permutation_seeds": PERMUTATION_MODEL_SEEDS,
    "target_permutations": N_TARGET_PERMUTATIONS,
    "kappa_grid": KAPPA_GRID,
    "external_opened": False,
    "retraining": False,
}, name="value")
display(configuration.to_frame())


## 2. Preflight de universo, firmas y artefactos

Esta celda impide tres errores frecuentes: usar un universo candidato
distinto, recortar un ranking que no llega al mayor K, o comparar el
observado 5/5 con permutaciones de dos semillas.


In [ ]:
CANDIDATE_FILE = (
    RESULTS_ROOT / "05_biology_validation" / "candidate_prior_for_null.csv.gz"
)
SIGNATURE_MANIFEST_FILE = (
    RESULTS_ROOT / "05_biology_validation" / "locked_signature_manifest.json"
)
candidates = pd.read_csv(CANDIDATE_FILE)
required_candidate_columns = {
    "candidate_edge_id", "source_idx", "target_idx", "source_gene",
    "target_gene", "prior_weight", "edge_type", "sign",
    "source_degree_bin", "target_degree_bin", "null_stratum",
}
missing = required_candidate_columns - set(candidates.columns)
if missing:
    raise ValueError(f"El prior candidato carece de {sorted(missing)}")

candidates = candidates.sort_values("candidate_edge_id").reset_index(drop=True)
candidates["candidate_edge_id"] = candidates.candidate_edge_id.astype(int)
if candidates.candidate_edge_id.duplicated().any():
    raise RuntimeError("candidate_edge_id contiene duplicados.")
for column in ("source_gene", "target_gene"):
    candidates[column] = candidates[column].astype(str).str.upper()
candidates["null_stratum"] = candidates.null_stratum.astype(str)
candidate_lookup = candidates.set_index("candidate_edge_id", drop=False)
candidate_ids = candidates.candidate_edge_id.to_numpy(np.int64)
n_candidate_edges = int(len(candidates))
dense_edge_capacity = int(candidate_ids.max()) + 1
candidate_id_to_row = np.full(dense_edge_capacity, -1, dtype=np.int64)
candidate_id_to_row[candidate_ids] = np.arange(n_candidate_edges, dtype=np.int64)
full_edge_index = np.full((2, dense_edge_capacity), -1, dtype=np.int64)
full_edge_index[:, candidate_ids] = candidates[
    ["source_idx", "target_idx"]
].to_numpy(np.int64).T

k_table = pd.DataFrame({"kappa": sorted(KAPPA_GRID)})
k_table["K"] = np.rint(k_table.kappa * n_candidate_edges).astype(int)
if k_table.K.duplicated().any():
    raise ValueError("Dos valores de kappa producen el mismo K.")
if k_table.K.max() >= n_candidate_edges:
    raise ValueError("El mayor K agota el universo candidato.")
K_VALUES = tuple(k_table.K.astype(int))
K_TO_KAPPA = dict(zip(k_table.K.astype(int), k_table.kappa.astype(float)))
K_MAX = int(max(K_VALUES))

directional_signatures, _, _, signature_source = load_locked_signature_resources(
    results_root=RESULTS_ROOT,
    resource_dir=RESOURCE_DIR,
)
directional_signatures = {
    str(class_name): {str(gene).upper() for gene in genes}
    for class_name, genes in directional_signatures.items()
}
if set(CLASS_NAMES) - set(directional_signatures):
    raise RuntimeError("Faltan firmas direccionales para alguna clase del endpoint.")
candidate_gene_universe = set(candidates.source_gene) | set(candidates.target_gene)
if not (directional_signatures[TARGET_CLASS] & candidate_gene_universe):
    raise RuntimeError("La firma Basal-like no intersecta el prior candidato.")


@lru_cache(maxsize=None)
def dense_metadata(path_text):
    path = Path(path_text)
    with np.load(path, allow_pickle=True) as dense:
        required = {
            "class_names", "structural_unit_id", "differential_logits",
            "context_score", "selection_frequency",
        }
        missing_dense = required - set(dense.files)
        if missing_dense:
            raise ValueError(f"{path}: faltan {sorted(missing_dense)}")
        class_names = tuple(map(str, dense["class_names"].tolist()))
        unit_id = np.asarray(dense["structural_unit_id"], dtype=np.int64)
        score_shape = tuple(np.asarray(dense["differential_logits"]).shape)
    if unit_id.size != dense_edge_capacity:
        raise RuntimeError(
            f"{path}: E_dense={unit_id.size}, "
            f"capacidad esperada={dense_edge_capacity}."
        )
    valid = candidate_ids[unit_id[candidate_ids] >= 0]
    units, counts = np.unique(unit_id[valid], return_counts=True)
    if not len(units):
        raise RuntimeError(f"{path}: no contiene unidades estructurales válidas.")
    if set(class_names) != set(CLASS_NAMES):
        raise RuntimeError(f"{path}: clases incompatibles {class_names}.")
    return {
        "path": str(path),
        "classes": class_names,
        "score_shape": score_shape,
        "n_valid_edges": int(len(valid)),
        "n_structural_units": int(len(units)),
        "unit_widths": tuple(map(int, np.unique(counts))),
        "supports_k_max": bool(len(units) >= K_MAX),
    }


observed_audit_rows = []
for seed in LOCKED_SEEDS:
    path = OBSERVED_DENSE_ROOT / f"seed_{seed}" / "transport_support_dense.npz"
    meta = dense_metadata(str(path))
    observed_audit_rows.append({"seed": seed, **meta})
observed_audit = pd.DataFrame(observed_audit_rows)
if not observed_audit.supports_k_max.all():
    raise RuntimeError("Algún artefacto observado no permite reconstruir K_max.")
if any(widths != (1,) for widths in observed_audit.unit_widths):
    raise RuntimeError(
        "Este análisis Exact-K requiere una arista dirigida por unidad estructural."
    )

contract_files = sorted(PERMUTATION_RESULT_ROOT.glob("permutation_*.json"))
permutation_audit_rows = []
for contract_file in contract_files:
    contract = json.loads(contract_file.read_text(encoding="utf-8"))
    permutation_id = int(contract["permutation_id"])
    if permutation_id >= N_TARGET_PERMUTATIONS:
        continue
    if bool(contract.get("external_opened", True)):
        raise RuntimeError(f"{contract_file}: external_opened no es False.")
    contract_seeds = tuple(map(int, contract.get("model_seeds", ())))
    if contract_seeds != PERMUTATION_MODEL_SEEDS:
        raise RuntimeError(
            f"{contract_file}: semillas {contract_seeds} != "
            f"{PERMUTATION_MODEL_SEEDS}."
        )
    artifact_root = ARTIFACTS_ROOT / (
        f"{PERMUTATION_ARTIFACT_PREFIX}{permutation_id:03d}"
    )
    dense_paths = [
        artifact_root / f"seed_{seed}" / "transport_support_dense.npz"
        for seed in PERMUTATION_MODEL_SEEDS
    ]
    row = {
        "permutation_id": permutation_id,
        "contract_file": str(contract_file),
        "artifact_root": str(artifact_root),
        "both_dense_files": all(path.is_file() for path in dense_paths),
        "supports_k_max": False,
    }
    if row["both_dense_files"]:
        metadata = [dense_metadata(str(path)) for path in dense_paths]
        row["supports_k_max"] = all(meta["supports_k_max"] for meta in metadata)
        row["minimum_rank_capacity"] = min(
            meta["n_structural_units"] for meta in metadata
        )
        widths = {meta["unit_widths"] for meta in metadata}
        if widths != {(1,)}:
            raise RuntimeError(
                f"Permutación {permutation_id}: anchura estructural {widths}."
            )
    else:
        row["minimum_rank_capacity"] = np.nan
    permutation_audit_rows.append(row)
permutation_audit = pd.DataFrame(permutation_audit_rows)
available_permutation_ids = tuple(
    sorted(
        permutation_audit.loc[
            permutation_audit.both_dense_files
            & permutation_audit.supports_k_max,
            "permutation_id",
        ].astype(int)
    )
) if len(permutation_audit) else ()

expected_ids = set(range(N_TARGET_PERMUTATIONS))
available_ids = set(available_permutation_ids)
missing_ids = sorted(expected_ids - available_ids)
if missing_ids:
    warnings.warn(
        f"Análisis formal todavía incompleto: disponibles "
        f"{len(available_ids)}/{N_TARGET_PERMUTATIONS} permutaciones con "
        f"ranking hasta K_max. Faltan {missing_ids}. El notebook producirá "
        "un resumen preliminar y se completará al volver a ejecutarlo."
    )

display(k_table)
display(observed_audit[[
    "seed", "n_valid_edges", "n_structural_units", "unit_widths",
    "supports_k_max",
]])
display(permutation_audit if len(permutation_audit) else pd.DataFrame({
    "estado": ["Aún no hay permutaciones resumidas"]
}))
print(
    f"Preflight: prior={n_candidate_edges:,} aristas | K_max={K_MAX:,} | "
    f"permutaciones utilizables={len(available_permutation_ids)}/"
    f"{N_TARGET_PERMUTATIONS}"
)


## 3. Funciones de reconstrucción y métricas

Cada corte se reconstruye desde `differential_logits`, `context_score` y
`selection_frequency`, con los pesos y el selector modular del protocolo.
La estabilidad de Nogueira corrige la selección esperada por azar; el
índice de Kuncheva hace la corrección de azar por pares cuando todos los
conjuntos tienen el mismo tamaño.

Para comparar firmas de tamaños distintos, el enriquecimiento se divide
por su expectativa exacta bajo el prior candidato, condicionada a la
composición de tipo, signo y bins de grado dirigido del grafo seleccionado.


In [ ]:
def observed_dense_path(seed):
    return (
        OBSERVED_DENSE_ROOT
        / f"seed_{int(seed)}"
        / "transport_support_dense.npz"
    )


def permutation_dense_path(permutation_id, seed):
    return (
        ARTIFACTS_ROOT
        / f"{PERMUTATION_ARTIFACT_PREFIX}{int(permutation_id):03d}"
        / f"seed_{int(seed)}"
        / "transport_support_dense.npz"
    )


def extract_exact_k_from_dense(dense_path, *, class_name, k_edges, seed):
    dense_path = Path(dense_path)
    k_edges = int(k_edges)
    meta = dense_metadata(str(dense_path))
    if not meta["supports_k_max"]:
        raise RuntimeError(f"{dense_path}: ranking insuficiente para K_max.")
    if meta["unit_widths"] != (1,):
        raise RuntimeError(f"{dense_path}: unidad estructural no unitaria.")

    with np.load(dense_path, allow_pickle=True) as dense:
        class_names = tuple(map(str, dense["class_names"].tolist()))
        class_id = class_names.index(str(class_name))
        unit_id = np.asarray(dense["structural_unit_id"], dtype=np.int64)
        valid = candidate_ids[unit_id[candidate_ids] >= 0]
        units, first_positions = np.unique(unit_id[valid], return_index=True)
        representative_ids = valid[first_positions]
        if len(units) < k_edges:
            raise RuntimeError(
                f"{dense_path}: {len(units)} unidades < K={k_edges}."
            )
        scores = combine_biology_scores(
            np.asarray(dense["differential_logits"], dtype=float),
            np.asarray(dense["context_score"], dtype=float),
            np.asarray(dense["selection_frequency"], dtype=float),
            context_weight=CONTEXT_WEIGHT,
            support_weight=SUPPORT_WEIGHT,
        )
        unit_scores = scores[class_id, representative_ids]
        selected_local = module_aware_exact_k(
            full_edge_index[:, representative_ids],
            unit_scores,
            k_edges=k_edges,
            pool_multiplier=MODULE_POOL_MULTIPLIER,
            target_components=MODULE_TARGET_COMPONENTS,
            min_component_edges=MODULE_MIN_EDGES,
        )
        selected_ids = representative_ids[selected_local]
        differential = np.asarray(
            dense["differential_logits"][class_id], dtype=float
        )
        context = np.asarray(dense["context_score"][class_id], dtype=float)
        support = np.asarray(
            dense["selection_frequency"][class_id], dtype=float
        )

    frame = candidate_lookup.loc[selected_ids].reset_index(drop=True).copy()
    frame["seed"] = int(seed)
    frame["class_name"] = str(class_name)
    frame["kappa"] = float(K_TO_KAPPA[k_edges])
    frame["K"] = k_edges
    frame["differential_logit"] = differential[selected_ids]
    frame["context_score"] = context[selected_ids]
    frame["selection_frequency"] = support[selected_ids]
    frame["biology_score"] = scores[class_id, selected_ids]
    frame = frame.sort_values(
        ["biology_score", "candidate_edge_id"],
        ascending=[False, True],
    ).reset_index(drop=True)
    frame.insert(0, "biology_rank", np.arange(1, len(frame) + 1))
    if len(frame) != k_edges or frame.candidate_edge_id.nunique() != k_edges:
        raise RuntimeError(
            f"{dense_path}/{class_name}: Exact-K roto ({len(frame)} != {k_edges})."
        )
    return frame


def strict_core(frames):
    if len(frames) < 2:
        raise ValueError("Se requieren al menos dos grafos.")
    id_sets = [set(frame.candidate_edge_id.astype(int)) for frame in frames]
    strict_ids = set.intersection(*id_sets)
    core = candidate_lookup.loc[sorted(strict_ids)].reset_index(drop=True).copy()
    core["selection_count"] = len(frames)
    return core


def selection_stability(frames, *, universe_size):
    if len(frames) < 2:
        raise ValueError("Se requieren al menos dos grafos.")
    sets = [set(frame.candidate_edge_id.astype(int)) for frame in frames]
    sizes = np.asarray([len(values) for values in sets], dtype=float)
    if len(set(sizes.astype(int))) != 1:
        raise RuntimeError("Kuncheva requiere conjuntos de igual tamaño.")
    k_edges = int(sizes[0])
    pair_rows = []
    for left, right in combinations(range(len(sets)), 2):
        intersection = len(sets[left] & sets[right])
        union = len(sets[left] | sets[right])
        expected = k_edges * k_edges / float(universe_size)
        denominator = k_edges - expected
        pair_rows.append({
            "jaccard": intersection / union if union else np.nan,
            "kuncheva": (
                (intersection - expected) / denominator
                if denominator > 0 else np.nan
            ),
        })

    selection = np.zeros((len(sets), int(universe_size)), dtype=np.uint8)
    for row_id, values in enumerate(sets):
        ids = np.fromiter(values, dtype=np.int64)
        rows = candidate_id_to_row[ids]
        if np.any(rows < 0):
            raise RuntimeError("Una selección contiene aristas fuera del prior.")
        selection[row_id, rows] = 1
    q_bar = float(selection.sum(axis=1).mean())
    selected_fraction = q_bar / float(universe_size)
    denominator = selected_fraction * (1.0 - selected_fraction)
    mean_feature_variance = float(
        selection.var(axis=0, ddof=1).mean()
    )
    nogueira = (
        1.0 - mean_feature_variance / denominator
        if denominator > 0 else np.nan
    )
    strict_ids = set.intersection(*sets)
    return {
        "K": k_edges,
        "n_seeds": len(sets),
        "mean_pairwise_jaccard": float(
            np.mean([row["jaccard"] for row in pair_rows])
        ),
        "minimum_pairwise_jaccard": float(
            np.min([row["jaccard"] for row in pair_rows])
        ),
        "mean_kuncheva": float(
            np.mean([row["kuncheva"] for row in pair_rows])
        ),
        "nogueira_stability": float(nogueira),
        "strict_edges": int(len(strict_ids)),
        "strict_fraction_of_K": float(len(strict_ids) / k_edges),
        "union_edges": int(len(set.union(*sets))),
    }


stratum_codes, stratum_labels = pd.factorize(
    candidates.null_stratum, sort=True
)
stratum_codes = np.asarray(stratum_codes, dtype=np.int64)
n_strata = int(stratum_codes.max()) + 1
stratum_sizes = np.bincount(stratum_codes, minlength=n_strata).astype(float)
source_genes = candidates.source_gene.to_numpy(str)
target_genes = candidates.target_gene.to_numpy(str)
signature_hits = {}
signature_pool_means = {}
for signature_name, members in directional_signatures.items():
    source_hit = np.isin(source_genes, list(members))
    target_hit = np.isin(target_genes, list(members))
    for statistic, hits in {
        "at_least_one_endpoint": source_hit | target_hit,
        "both_endpoints": source_hit & target_hit,
    }.items():
        key = (signature_name, statistic)
        hits = hits.astype(float)
        signature_hits[key] = hits
        signature_pool_means[key] = (
            np.bincount(
                stratum_codes,
                weights=hits,
                minlength=n_strata,
            )
            / stratum_sizes
        )


def matched_signature_enrichment(selected):
    ids = selected.candidate_edge_id.astype(int).to_numpy()
    if not len(ids):
        raise ValueError("No puede evaluarse un grafo vacío.")
    selected_rows = candidate_id_to_row[ids]
    if np.any(selected_rows < 0):
        raise RuntimeError("El grafo contiene aristas fuera del prior candidato.")
    selected_codes = stratum_codes[selected_rows]
    rows = []
    for signature_name in CLASS_NAMES:
        for statistic in ("at_least_one_endpoint", "both_endpoints"):
            key = (signature_name, statistic)
            observed = float(signature_hits[key][selected_rows].mean())
            expected = float(
                signature_pool_means[key][selected_codes].mean()
            )
            fold = observed / expected if expected > 0 else np.nan
            rows.append({
                "signature": signature_name,
                "statistic": statistic,
                "observed_fraction": observed,
                "matched_expected_fraction": expected,
                "fold_over_matched_expectation": fold,
                "log2_fold": (
                    float(np.log2(fold)) if fold > 0 else -np.inf
                ),
                "n_edges": int(len(ids)),
            })
    return pd.DataFrame(rows)


def specificity_summary(selected):
    if selected.empty:
        empty_specificity = pd.DataFrame([
            {
                "statistic": statistic,
                "basal_log2_fold": -np.inf,
                "best_alternative_signature": None,
                "best_alternative_log2_fold": np.nan,
                "specificity_margin": -np.inf,
                "n_edges": 0,
            }
            for statistic in ("at_least_one_endpoint", "both_endpoints")
        ])
        empty_enrichment = pd.DataFrame([
            {
                "signature": signature_name,
                "statistic": statistic,
                "observed_fraction": np.nan,
                "matched_expected_fraction": np.nan,
                "fold_over_matched_expectation": np.nan,
                "log2_fold": -np.inf,
                "n_edges": 0,
            }
            for signature_name in CLASS_NAMES
            for statistic in ("at_least_one_endpoint", "both_endpoints")
        ])
        return empty_specificity, empty_enrichment
    enrichment = matched_signature_enrichment(selected)
    rows = []
    for statistic, block in enrichment.groupby("statistic", sort=False):
        basal = float(
            block.loc[block.signature.eq(TARGET_CLASS), "log2_fold"].iloc[0]
        )
        alternatives = block.loc[
            ~block.signature.eq(TARGET_CLASS),
            ["signature", "log2_fold"],
        ].sort_values("log2_fold", ascending=False)
        best = alternatives.iloc[0]
        rows.append({
            "statistic": statistic,
            "basal_log2_fold": basal,
            "best_alternative_signature": str(best.signature),
            "best_alternative_log2_fold": float(best.log2_fold),
            "specificity_margin": basal - float(best.log2_fold),
            "n_edges": int(len(selected)),
        })
    return pd.DataFrame(rows), enrichment


## 4. Curvas observadas de cinco semillas

Este bloque genera de nuevo los cuatro grafos de clase para cada semilla
y cada K, sus consensos Exact-K y sus núcleos estrictos 5/5. Sus resultados
describen reproducibilidad; no se usan como observado frente al nulo de
etiquetas de dos semillas.


In [ ]:
observed_graphs = {}
structural_rows = []
topology_rows = []
biology_rows = []
specificity_rows = []

for k_edges in K_VALUES:
    kappa = K_TO_KAPPA[k_edges]
    print(f"[observado] kappa={kappa:.3f} | K={k_edges}")
    consensus_by_class = {}
    for class_name in CLASS_NAMES:
        seed_graphs = {}
        class_seed_dir = (
            SEED_EXPORT_DIR
            / f"kappa_{kappa:.3f}_K_{k_edges}"
            / safe_name(class_name)
        )
        class_seed_dir.mkdir(parents=True, exist_ok=True)
        for seed in LOCKED_SEEDS:
            frame = extract_exact_k_from_dense(
                observed_dense_path(seed),
                class_name=class_name,
                k_edges=k_edges,
                seed=seed,
            )
            observed_graphs[(k_edges, class_name, seed)] = frame
            seed_graphs[seed] = frame
            frame.to_csv(
                class_seed_dir / f"seed_{seed}.csv.gz",
                index=False,
                compression="gzip",
            )

        consensus, strict, aggregate_summary = aggregate_seed_graphs(
            seed_graphs,
            k_edges=k_edges,
        )
        consensus["class_name"] = class_name
        consensus["kappa"] = kappa
        consensus["K"] = k_edges
        strict["class_name"] = class_name
        strict["kappa"] = kappa
        strict["K"] = k_edges
        consensus_by_class[class_name] = consensus

        consensus.to_csv(
            CONSENSUS_EXPORT_DIR
            / f"consensus_{safe_name(class_name)}_K_{k_edges}.csv",
            index=False,
        )
        strict.to_csv(
            CONSENSUS_EXPORT_DIR
            / f"strict_5of5_{safe_name(class_name)}_K_{k_edges}.csv",
            index=False,
        )

        stability = selection_stability(
            list(seed_graphs.values()),
            universe_size=n_candidate_edges,
        )
        structural_rows.append({
            "class_name": class_name,
            "kappa": kappa,
            **stability,
        })
        for graph_kind, graph in (("consensus", consensus), ("strict_5of5", strict)):
            topology_rows.append({
                "class_name": class_name,
                "kappa": kappa,
                "K": k_edges,
                "graph_kind": graph_kind,
                **topology_metrics(graph),
            })

        enrichment = matched_signature_enrichment(consensus)
        enrichment.insert(0, "graph_class", class_name)
        enrichment.insert(0, "K", k_edges)
        enrichment.insert(0, "kappa", kappa)
        biology_rows.append(enrichment)

        specificity, _ = specificity_summary(consensus)
        specificity.insert(0, "graph_class", class_name)
        specificity.insert(0, "K", k_edges)
        specificity.insert(0, "kappa", kappa)
        specificity_rows.append(specificity)

observed_structural = pd.DataFrame(structural_rows)
observed_topology = pd.DataFrame(topology_rows)
observed_biology = pd.concat(biology_rows, ignore_index=True)
observed_specificity = pd.concat(specificity_rows, ignore_index=True)

observed_structural.to_csv(
    OUTPUT_DIR / "observed_five_seed_structural_stability.csv", index=False
)
observed_topology.to_csv(
    OUTPUT_DIR / "observed_five_seed_topology.csv", index=False
)
observed_biology.to_csv(
    OUTPUT_DIR / "observed_consensus_signature_enrichment.csv", index=False
)
observed_specificity.to_csv(
    OUTPUT_DIR / "observed_consensus_signature_specificity.csv", index=False
)

display(
    observed_structural.loc[
        observed_structural.class_name.eq(TARGET_CLASS),
        [
            "kappa", "K", "mean_pairwise_jaccard", "mean_kuncheva",
            "nogueira_stability", "strict_edges", "strict_fraction_of_K",
        ],
    ]
)
display(
    observed_specificity.loc[
        observed_specificity.graph_class.eq(TARGET_CLASS)
        & observed_specificity.statistic.eq("at_least_one_endpoint")
    ]
)


In [ ]:
palette = {
    "LumA": "#2f6db3",
    "LumB": "#168c8c",
    "HER2-enriched": "#7a3e9d",
    "Basal-like": "#d73027",
}
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.3), constrained_layout=True)

for class_name in CLASS_NAMES:
    block = observed_structural.loc[
        observed_structural.class_name.eq(class_name)
    ].sort_values("kappa")
    axes[0, 0].plot(
        block.kappa, block.nogueira_stability,
        marker="o", color=palette[class_name], label=class_name,
    )
    axes[0, 1].plot(
        block.kappa, block.strict_fraction_of_K,
        marker="o", color=palette[class_name], label=class_name,
    )
axes[0, 0].set(title="A  Estabilidad corregida por azar", ylabel="Nogueira")
axes[0, 1].set(title="B  Núcleo reproducido en 5/5 semillas", ylabel="|núcleo 5/5| / K")

basal_biology = observed_biology.loc[
    observed_biology.graph_class.eq(TARGET_CLASS)
    & observed_biology.signature.eq(TARGET_CLASS)
].sort_values("kappa")
for statistic, label, marker in (
    ("at_least_one_endpoint", "≥1 extremo Basal-like", "o"),
    ("both_endpoints", "ambos extremos Basal-like", "s"),
):
    block = basal_biology.loc[basal_biology.statistic.eq(statistic)]
    axes[1, 0].plot(
        block.kappa, block.fold_over_matched_expectation,
        marker=marker, linewidth=2.2, label=label,
    )
axes[1, 0].axhline(1.0, color="#777777", linestyle="--", linewidth=1)
axes[1, 0].set(
    title="C  Recuperación Basal-like en el consenso",
    ylabel="Fold / expectativa emparejada",
)

basal_specificity = observed_specificity.loc[
    observed_specificity.graph_class.eq(TARGET_CLASS)
    & observed_specificity.statistic.eq("at_least_one_endpoint")
].sort_values("kappa")
axes[1, 1].plot(
    basal_specificity.kappa,
    basal_specificity.specificity_margin,
    marker="o", linewidth=2.2, color=palette[TARGET_CLASS],
)
axes[1, 1].axhline(0.0, color="#777777", linestyle="--", linewidth=1)
axes[1, 1].set(
    title="D  Especificidad Basal-like",
    ylabel="Margen log2-fold frente a la mejor alternativa",
)

for axis in axes.flat:
    axis.set_xlabel("κ")
    axis.set_xticks(sorted(KAPPA_GRID))
    axis.grid(axis="y", color="#dddddd", linewidth=0.7)
axes[0, 0].legend(frameon=False, ncol=2)
axes[1, 0].legend(frameon=False)
fig.suptitle("Sensibilidad al presupuesto Exact-K", fontsize=15, fontweight="bold")
for extension in ("png", "pdf", "svg"):
    fig.savefig(
        FIGURE_DIR / f"Supplementary_K_budget_sensitivity.{extension}",
        dpi=350 if extension == "png" else None,
        bbox_inches="tight",
    )
plt.show()


## 5. Observado emparejado y permutaciones de etiquetas

La comparación formal usa el núcleo estricto 2/2 de las semillas
declaradas en el protocolo, tanto para el observado como para cada
permutación. Esto evita atribuir al modelo una ventaja artificial por
disponer de cinco reentrenamientos observados frente a dos nulos.

Para cada K se calcula

\[
S(K)=\log_2 FE_{Basal}(K)-
\max_{c\neq Basal}\log_2 FE_c(K),
\]

usando aristas con al menos un extremo en cada firma y una expectativa
emparejada por tipo, signo y bins de grado dirigido. El contraste
principal es \(T_{rob}=\min_K S(K)\): solo es alto si la especificidad
Basal-like se conserva en toda la rejilla.


In [ ]:
paired_observed_rows = []
for k_edges in K_VALUES:
    frames = [
        observed_graphs[(k_edges, TARGET_CLASS, seed)]
        for seed in PERMUTATION_MODEL_SEEDS
    ]
    core = strict_core(frames)
    specificity, enrichment = specificity_summary(core)
    specificity["kappa"] = K_TO_KAPPA[k_edges]
    specificity["K"] = k_edges
    specificity["strict_edges_2of2"] = len(core)
    paired_observed_rows.append(specificity)

paired_observed_curve = pd.concat(paired_observed_rows, ignore_index=True)
paired_observed_curve.to_csv(
    OUTPUT_DIR / "observed_same_two_seeds_k_curve.csv", index=False
)

observed_primary_curve = paired_observed_curve.loc[
    paired_observed_curve.statistic.eq("at_least_one_endpoint")
].sort_values("K")
observed_secondary_curve = paired_observed_curve.loc[
    paired_observed_curve.statistic.eq("both_endpoints")
].sort_values("K")
T_OBS_PRIMARY = float(observed_primary_curve.specificity_margin.min())
T_OBS_SECONDARY = float(observed_secondary_curve.basal_log2_fold.min())

display(paired_observed_curve)
print(f"T_obs primario (peor margen Basal-like) = {T_OBS_PRIMARY:.4f}")
print(f"T_obs secundario (peor log2-fold ambos extremos) = {T_OBS_SECONDARY:.4f}")


In [ ]:
permutation_curve_rows = []
permutation_robust_rows = []

for position, permutation_id in enumerate(available_permutation_ids, start=1):
    if position == 1 or position % 5 == 0 or position == len(available_permutation_ids):
        print(
            f"[recorte permutaciones] {position}/"
            f"{len(available_permutation_ids)} | id={permutation_id:03d}"
        )
    current_rows = []
    for k_edges in K_VALUES:
        permuted_frames = [
            extract_exact_k_from_dense(
                permutation_dense_path(permutation_id, seed),
                class_name=TARGET_CLASS,
                k_edges=k_edges,
                seed=seed,
            )
            for seed in PERMUTATION_MODEL_SEEDS
        ]
        core = strict_core(permuted_frames)
        specificity, _ = specificity_summary(core)
        specificity["permutation_id"] = permutation_id
        specificity["kappa"] = K_TO_KAPPA[k_edges]
        specificity["K"] = k_edges
        specificity["strict_edges_2of2"] = len(core)
        current_rows.append(specificity)
    current = pd.concat(current_rows, ignore_index=True)
    permutation_curve_rows.append(current)
    primary = current.loc[
        current.statistic.eq("at_least_one_endpoint")
    ]
    secondary = current.loc[current.statistic.eq("both_endpoints")]
    permutation_robust_rows.append({
        "permutation_id": permutation_id,
        "T_primary_min_specificity_margin": float(
            primary.specificity_margin.min()
        ),
        "T_secondary_min_basal_log2_fold_both": float(
            secondary.basal_log2_fold.min()
        ),
        "minimum_strict_edges_2of2": int(current.strict_edges_2of2.min()),
    })

permutation_curves = (
    pd.concat(permutation_curve_rows, ignore_index=True)
    if permutation_curve_rows else pd.DataFrame()
)
permutation_robust = pd.DataFrame(permutation_robust_rows)
permutation_curves.to_csv(
    OUTPUT_DIR / "label_permutation_k_curves.csv", index=False
)
permutation_robust.to_csv(
    OUTPUT_DIR / "label_permutation_robust_statistics.csv", index=False
)

B = int(len(permutation_robust))
FORMAL_COMPLETE = B == N_TARGET_PERMUTATIONS
if B:
    p_primary = float(
        (1 + np.sum(
            permutation_robust.T_primary_min_specificity_margin
            .to_numpy(float) >= T_OBS_PRIMARY
        )) / (B + 1)
    )
    p_secondary = float(
        (1 + np.sum(
            permutation_robust.T_secondary_min_basal_log2_fold_both
            .to_numpy(float) >= T_OBS_SECONDARY
        )) / (B + 1)
    )
else:
    p_primary = np.nan
    p_secondary = np.nan

formal_summary = pd.DataFrame([
    {
        "analysis_status": (
            "complete_49_permutations"
            if FORMAL_COMPLETE else "preliminary_incomplete_permutations"
        ),
        "statistic": "T_primary_min_specificity_margin",
        "observed": T_OBS_PRIMARY,
        "empirical_p_upper": p_primary,
        "completed_permutations": B,
        "target_permutations": N_TARGET_PERMUTATIONS,
        "minimum_attainable_p": 1 / (B + 1) if B else np.nan,
        "observed_seeds": str(PERMUTATION_MODEL_SEEDS),
        "permutation_seeds": str(PERMUTATION_MODEL_SEEDS),
    },
    {
        "analysis_status": (
            "complete_49_permutations"
            if FORMAL_COMPLETE else "preliminary_incomplete_permutations"
        ),
        "statistic": "T_secondary_min_basal_log2_fold_both",
        "observed": T_OBS_SECONDARY,
        "empirical_p_upper": p_secondary,
        "completed_permutations": B,
        "target_permutations": N_TARGET_PERMUTATIONS,
        "minimum_attainable_p": 1 / (B + 1) if B else np.nan,
        "observed_seeds": str(PERMUTATION_MODEL_SEEDS),
        "permutation_seeds": str(PERMUTATION_MODEL_SEEDS),
    },
])
formal_summary.to_csv(
    OUTPUT_DIR / "robust_k_permutation_summary.csv", index=False
)

per_k_rows = []
if B:
    for statistic, metric in (
        ("at_least_one_endpoint", "specificity_margin"),
        ("both_endpoints", "basal_log2_fold"),
    ):
        for k_edges in K_VALUES:
            observed_value = float(
                paired_observed_curve.loc[
                    paired_observed_curve.statistic.eq(statistic)
                    & paired_observed_curve.K.eq(k_edges),
                    metric,
                ].iloc[0]
            )
            null_values = permutation_curves.loc[
                permutation_curves.statistic.eq(statistic)
                & permutation_curves.K.eq(k_edges),
                metric,
            ].to_numpy(float)
            per_k_rows.append({
                "statistic": statistic,
                "metric": metric,
                "kappa": K_TO_KAPPA[k_edges],
                "K": k_edges,
                "observed": observed_value,
                "permutation_mean": float(np.mean(null_values)),
                "permutation_q025": float(np.quantile(null_values, 0.025)),
                "permutation_q975": float(np.quantile(null_values, 0.975)),
                "empirical_p_upper_exploratory": float(
                    (1 + np.sum(null_values >= observed_value)) / (B + 1)
                ),
                "completed_permutations": B,
            })
per_k_summary = pd.DataFrame(per_k_rows)
per_k_summary.to_csv(
    OUTPUT_DIR / "exploratory_per_k_permutation_summary.csv", index=False
)

display(formal_summary)
if not FORMAL_COMPLETE:
    print(
        "El valor p mostrado es preliminar. Para el resultado final, vuelva a "
        "ejecutar este notebook cuando estén disponibles las 49 permutaciones."
    )


In [ ]:
if B:
    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6), constrained_layout=True)
    for permutation_id, block in permutation_curves.loc[
        permutation_curves.statistic.eq("at_least_one_endpoint")
    ].groupby("permutation_id"):
        plot_values = block.specificity_margin.replace(
            [np.inf, -np.inf], np.nan
        )
        axes[0].plot(
            block.kappa, plot_values,
            color="#bdbdbd", alpha=0.35, linewidth=0.8,
        )
    observed_plot_values = observed_primary_curve.specificity_margin.replace(
        [np.inf, -np.inf], np.nan
    )
    axes[0].plot(
        observed_primary_curve.kappa,
        observed_plot_values,
        color="#d73027", marker="o", linewidth=2.5, label="Observado",
    )
    axes[0].axhline(0, color="#666666", linestyle="--", linewidth=1)
    axes[0].set(
        title="A  Margen Basal-like a través de K",
        xlabel="κ",
        ylabel="Margen log2-fold emparejado",
    )
    axes[0].legend(frameon=False)

    null = permutation_robust.T_primary_min_specificity_margin.to_numpy(float)
    finite_null = null[np.isfinite(null)]
    if len(finite_null) and np.isfinite(T_OBS_PRIMARY):
        axes[1].hist(
            finite_null,
            bins=min(15, max(5, len(finite_null) // 3)),
            color="#bdbdbd",
            edgecolor="white",
        )
        axes[1].axvline(
            T_OBS_PRIMARY, color="#d73027", linewidth=2.5,
            label=f"Observado = {T_OBS_PRIMARY:.3f}",
        )
        axes[1].legend(frameon=False)
    else:
        axes[1].text(
            0.5, 0.5,
            "Sin valores finitos:\nalgún núcleo 2/2 quedó vacío",
            ha="center", va="center", transform=axes[1].transAxes,
        )
    axes[1].set(
        title="B  Nulo de etiquetas del peor K",
        xlabel="Trob = minK S(K)",
        ylabel="Permutaciones",
    )
    status = "completo" if FORMAL_COMPLETE else f"preliminar, B={B}"
    fig.suptitle(
        f"Robustez de la especificidad Basal-like ({status}; p={p_primary:.3g})",
        fontsize=14,
        fontweight="bold",
    )
    for extension in ("png", "pdf", "svg"):
        fig.savefig(
            FIGURE_DIR / f"Supplementary_K_robust_permutation.{extension}",
            dpi=350 if extension == "png" else None,
            bbox_inches="tight",
        )
    plt.show()
else:
    print("Aún no hay permutaciones densas utilizables; se omite esta figura.")


## 6. Resumen reproducible y texto de resultados

El texto generado distingue automáticamente un análisis completo de uno
preliminar. No copie un valor *p* al manuscrito hasta que el estado sea
`complete_49_permutations`.


In [ ]:
basal_structural = observed_structural.loc[
    observed_structural.class_name.eq(TARGET_CLASS)
].sort_values("K")
basal_consensus_biology = observed_biology.loc[
    observed_biology.graph_class.eq(TARGET_CLASS)
    & observed_biology.signature.eq(TARGET_CLASS)
]
basal_at_least = basal_consensus_biology.loc[
    basal_consensus_biology.statistic.eq("at_least_one_endpoint")
].sort_values("K")
basal_both = basal_consensus_biology.loc[
    basal_consensus_biology.statistic.eq("both_endpoints")
].sort_values("K")
basal_margin = observed_specificity.loc[
    observed_specificity.graph_class.eq(TARGET_CLASS)
    & observed_specificity.statistic.eq("at_least_one_endpoint")
].sort_values("K")

status_sentence = (
    f"La prueba robusta basada en 49 permutaciones produjo "
    f"Trob={T_OBS_PRIMARY:.3f} y p={p_primary:.3g}."
    if FORMAL_COMPLETE
    else (
        f"La prueba por permutación permanece preliminar "
        f"({B}/{N_TARGET_PERMUTATIONS} permutaciones disponibles) y no se "
        "incorpora todavía como resultado confirmatorio."
    )
)
report_text = dedent(f'''\
# Sensibilidad del presupuesto Exact-K

Se evaluó una rejilla de cuatro densidades (κ={min(KAPPA_GRID):.3f}–{max(KAPPA_GRID):.3f}; K={min(K_VALUES)}–{max(K_VALUES)} aristas) reconstruyendo cada red desde las puntuaciones densas bloqueadas. En el grafo Basal-like, el Jaccard medio entre cinco semillas varió entre {basal_structural.mean_pairwise_jaccard.min():.3f} y {basal_structural.mean_pairwise_jaccard.max():.3f}, y la estabilidad de Nogueira entre {basal_structural.nogueira_stability.min():.3f} y {basal_structural.nogueira_stability.max():.3f}. La fracción del núcleo 5/5 fue {basal_structural.strict_fraction_of_K.min():.3f}–{basal_structural.strict_fraction_of_K.max():.3f}.

La recuperación de la firma Basal-like en los consensos permaneció en {basal_at_least.fold_over_matched_expectation.min():.2f}–{basal_at_least.fold_over_matched_expectation.max():.2f} veces la expectativa emparejada para al menos un extremo y {basal_both.fold_over_matched_expectation.min():.2f}–{basal_both.fold_over_matched_expectation.max():.2f} veces para ambos extremos. El margen de especificidad frente a la firma alternativa más enriquecida fue {basal_margin.specificity_margin.min():.3f}–{basal_margin.specificity_margin.max():.3f} en escala log2. {status_sentence}

Interpretación: las conclusiones se consideran robustas al presupuesto cuando la estabilidad no muestra un colapso localizado, los folds Basal-like se mantienen por encima de 1 y el margen de especificidad es positivo en los cuatro valores de K. La prueba formal exige además un Trob positivo y un valor p empírico compatible con el umbral inferencial definido para el manuscrito.
''').strip() + "\n"
(OUTPUT_DIR / "manuscript_ready_K_sensitivity.md").write_text(
    report_text, encoding="utf-8"
)

manifest = {
    "analysis": "post_training_exact_k_budget_sensitivity",
    "analysis_status": (
        "complete_49_permutations"
        if FORMAL_COMPLETE else "preliminary_incomplete_permutations"
    ),
    "external_opened": False,
    "model_refitted": False,
    "candidate_edges": n_candidate_edges,
    "kappa_grid": list(map(float, sorted(KAPPA_GRID))),
    "K_grid": list(map(int, K_VALUES)),
    "five_seed_descriptive": list(map(int, LOCKED_SEEDS)),
    "paired_permutation_seeds": list(map(int, PERMUTATION_MODEL_SEEDS)),
    "completed_permutations": B,
    "target_permutations": N_TARGET_PERMUTATIONS,
    "primary_statistic": "min_K matched-log2-fold specificity margin for Basal-like",
    "empirical_p_formula": "(1 + count(T_perm >= T_observed)) / (B + 1)",
    "input_hashes": {
        "protocol": sha256_file(PROTOCOL_FILE),
        "candidate_prior": sha256_file(CANDIDATE_FILE),
        "signature_manifest": sha256_file(SIGNATURE_MANIFEST_FILE),
    },
    "score_weights": {
        "context": CONTEXT_WEIGHT,
        "support": SUPPORT_WEIGHT,
    },
    "module_selector": {
        "pool_multiplier": MODULE_POOL_MULTIPLIER,
        "target_components": MODULE_TARGET_COMPONENTS,
        "minimum_edges": MODULE_MIN_EDGES,
    },
}
(OUTPUT_DIR / "ANALYSIS_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
(OUTPUT_DIR / "permutation_artifact_preflight.csv").write_text(
    permutation_audit.to_csv(index=False), encoding="utf-8"
)

display(Markdown(report_text))
print("\nArchivos generados:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(OUTPUT_DIR))


## Referencias metodológicas

- Nogueira S, Sechidis K, Brown G. *On the Stability of Feature
  Selection Algorithms*. Journal of Machine Learning Research. 2018;
  18(174):1–54. [Artículo](https://jmlr.org/papers/v18/17-514.html).
- Kuncheva LI. *A stability index for feature selection*. Proceedings
  of the 25th IASTED International Multi-Conference: Artificial
  Intelligence and Applications. 2007:390–395.
  [Registro bibliográfico](https://research.bangor.ac.uk/en/publications/a-stability-index-for-feature-selection/).
- Liu H, Roeder K, Wasserman L. *Stability Approach to Regularization
  Selection (StARS) for High Dimensional Graphical Models*. NeurIPS.
  2010. [Texto completo](https://pmc.ncbi.nlm.nih.gov/articles/PMC4138724/).
- Phipson B, Smyth GK. *Permutation P-values Should Never Be Zero:
  Calculating Exact P-values When Permutations Are Randomly Drawn*.
  Statistical Applications in Genetics and Molecular Biology. 2010;
  9:Article 39. doi:10.2202/1544-6115.1585.
  [PubMed](https://pubmed.ncbi.nlm.nih.gov/21044043/).

StARS se cita como fundamento del principio de elegir sparsidad junto a
reproducibilidad; este cuaderno no afirma implementar el algoritmo StARS
para un modelo gráfico gaussiano.
